In [1]:
%run start
%load_ext autoreload
%autoreload 2

Root set to: /home/bdudas/obesity_challange


In [23]:
from src.data.perturbation_data import get_loaders
from src.models.CycleTransformerv2 import CycleTransformer
from omegaconf import OmegaConf
import torch
import numpy as np

In [ ]:
modelwkgs = OmegaConf.load("configs/cycle_transformer.yaml")
cpkt_name = "cycle_vit"
pretrained_path = f"misc/best_runs/{cpkt_name}/checkpoints/best-checkpoint.ckpt"
model = CycleTransformer(**modelwkgs.model_kwargs)
model.configure_cycle()
model.load_state_dict(torch.load(pretrained_path)["state_dict"])

In [24]:
pertList = np.loadtxt("data/predict_perturbations.txt", dtype=str)

In [15]:
train_loader, val_loader,gene_to_idx, idx_to_gene = get_loaders("",batch_size=64, num_workers=4)

In [16]:
batch = next(iter(val_loader))
_, x_ctrl, _, _, ctrl_state = batch

In [30]:
samplePert = pertList[0]
pert_idx = gene_to_idx[samplePert]
pert_state = torch.tensor([pert_idx]*x_ctrl.size(0))

In [32]:
x_ctrl.shape

torch.Size([64, 32, 675])

In [31]:
z_ctrl = model.encoder(x_ctrl)
if pert_state.dim() > 1 and pert_state.size(1) > 1:
    pert_indices = torch.argmax(pert_state, dim=1)
else:
    pert_indices = pert_state.long()
z_prompt = model.get_perturbation_prompt(x_ctrl,pert_indices)
delta_fwd = model.transition_fwd(z_ctrl, z_prompt)
z_fake_pert = z_ctrl + delta_fwd
logits_fake_pert = model.latentClassifier(z_fake_pert)

IndexError: index 9302 is out of bounds for dimension 1 with size 32